## 📦 Imports & Setup

In [2]:
from __future__ import annotations
from pathlib import Path
from typing import List, Tuple, Dict, Any, Set, Optional
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import sys
import time
from matplotlib.colors import ListedColormap
from IPython.display import Video, display
import os

# ---------------------------------------------------------------------
# Data Parsing and Pre-processing (From CSV to NPZ)
# ---------------------------------------------------------------------

def parse_atom_indices_to_coords(indices_raw: any, grid_size: int) -> List[Tuple[int, int]]:
    """Parses a string of atom indices into a list of (x, y) coordinates."""
    if pd.isna(indices_raw) or str(indices_raw).strip() in ["", '""', "''"]:
        return []
    s = str(indices_raw).strip('"\' ')
    if not s:
        return []
    try:
        # Handle both comma and semicolon separators
        if ',' in s:
            parts = s.split(',')
        elif ';' in s:
            parts = s.split(';')
        else:
            parts = [s]
            
        indices = []
        for x in parts:
            x = x.strip()
            if x:
                try:
                    idx = int(x)
                    if 0 <= idx < grid_size * grid_size:
                        indices.append(idx)
                    else:
                        print(f"Warning: Index {idx} out of bounds for grid size {grid_size}", file=sys.stderr)
                except ValueError:
                    print(f"Warning: Could not parse index '{x}'", file=sys.stderr)
                    continue
        
        return [(idx % grid_size, idx // grid_size) for idx in indices]
    except Exception as e:
        print(f"Error parsing atom indices: {e}", file=sys.stderr)
        return []

def parse_grid_config_string_to_species_array(config_raw: any, N: int) -> Optional[np.ndarray]:
    """Parses a grid configuration string directly into a NumPy species array."""
    if pd.isna(config_raw) or str(config_raw).strip() in ["", '""', "''"]:
        return None
    s = str(config_raw).strip('"\' ')
    if not s:  # Additional check for empty string after stripping
        return None
    parts = s.split(';')
    if len(parts) != N * N:
        print(f"Warning: Expected {N*N} cells but found {len(parts)} in config string", file=sys.stderr)
        return None
    species = np.zeros((N, N), dtype=np.int8)
    try:
        for i, cell_data_str in enumerate(parts):
            if not cell_data_str.strip(): 
                continue
            # More robust parsing - handle potential malformed data
            cell_parts = cell_data_str.split('|')
            if len(cell_parts) == 0:
                continue
            species_val = int(cell_parts[0])
            # Validate species value is within expected range
            if species_val not in [0, 1]:
                print(f"Warning: Unexpected species value {species_val} at position {i}", file=sys.stderr)
            y, x = i // N, i % N
            species[y, x] = species_val
    except (ValueError, IndexError) as e:
        print(f"Error parsing grid config at position {i if 'i' in locals() else 'unknown'}: {e}", file=sys.stderr)
        return None
    return species

def preprocess_csv_to_npz(csv_file: Path, start_iteration: int, end_iteration: int) -> Optional[Path]:
    """
    Reads a specific iteration range from a CSV, processes it into NumPy arrays,
    and saves it to a uniquely named .npz file.
    """
    npz_file_name = f"{csv_file.stem}_iter_{start_iteration}_to_{end_iteration}.npz"
    npz_path = csv_file.parent / npz_file_name
    print(f"Pre-processing {csv_file.name} (iterations {start_iteration}-{end_iteration}) -> {npz_path.name}", file=sys.stderr)

    try:
        df_for_n = pd.read_csv(csv_file, dtype=str, usecols=["GridConfigStarting"], nrows=1)
        if df_for_n.empty:
            print(f"Error: CSV file {csv_file.name} appears to be empty.", file=sys.stderr)
            return None
        config_str = df_for_n["GridConfigStarting"].iloc[0]
        num_cells = len(str(config_str).strip('"\' ').split(';'))
        N = int(math.sqrt(num_cells))
        if N * N != num_cells:
            print(f"Error: Could not determine a square grid size from CSV.", file=sys.stderr)
            return None

        nrows_to_read = end_iteration - start_iteration + 1
        if nrows_to_read <= 0:
            print(f"No iterations in range {start_iteration}-{end_iteration}. Creating empty NPZ.", file=sys.stderr)
            np.savez_compressed(npz_path, N=N, data=np.array([], dtype=object))
            return npz_path

        df_sim = pd.read_csv(
            csv_file,
            dtype=str,
            skiprows=range(1, start_iteration + 1),
            nrows=nrows_to_read
        )
    except FileNotFoundError:
        print(f"Error: CSV file not found at {csv_file}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"Error reading CSV {csv_file}: {e}", file=sys.stderr)
        return None

    # Check if DataFrame has data
    if df_sim.empty:
        print(f"Warning: No data found in iteration range {start_iteration}-{end_iteration}", file=sys.stderr)
        np.savez_compressed(npz_path, N=N, states=np.array([], dtype=np.int8).reshape(0, N, N))
        return npz_path

    # Pre-calculate all states to avoid parsing during animation
    all_states = [parse_grid_config_string_to_species_array(row["GridConfigStarting"], N) for _, row in df_sim.iterrows()]
    
    # Add the final state from the last potential config (if it exists)
    if "GridConfigPotential" in df_sim.columns and not df_sim.empty:
        final_potential = parse_grid_config_string_to_species_array(df_sim.iloc[-1]["GridConfigPotential"], N)
        if final_potential is not None:
            all_states.append(final_potential)

    valid_states = [s for s in all_states if s is not None]

    if len(valid_states) == 0:
        print(f"Warning: No valid states found in the data", file=sys.stderr)
        np.savez_compressed(npz_path, N=N, states=np.array([], dtype=np.int8).reshape(0, N, N))
        return npz_path

    np.savez_compressed(npz_path, N=N, states=np.array(valid_states, dtype=np.int8))
    print(f"Successfully pre-processed and saved to {npz_path}", file=sys.stderr)
    return npz_path

# ---------------------------------------------------------------------
# Hyper-Optimized Animation Function
# ---------------------------------------------------------------------

def create_grid_animation_hyper_optimized(npz_file: Path, output_filename: str, fps: int, dpi: int, start_iteration_offset: int):
    """
    Creates an animation using the most direct and efficient rendering method possible.
    It calculates state differences and only updates artists that have changed.
    """
    print(f"Creating hyper-optimized animation for {npz_file.name}", file=sys.stderr)
    try:
        with np.load(npz_file) as data:
            N = data['N']
            states = data['states']
    except Exception as e:
        print(f"Error loading .npz file {npz_file}: {e}", file=sys.stderr)
        return

    if states.size == 0 or len(states) <= 1:
        print("Not enough states to animate.", file=sys.stderr)
        return

    total_frames = len(states) - 1
    if total_frames <= 0:
        print("No frame transitions to animate.", file=sys.stderr)
        return
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set(xlim=(-0.5, N - 0.5), ylim=(-0.5, N - 0.5), aspect="equal", xticks=[], yticks=[])
    fig.tight_layout()

    cmap = ListedColormap(['#d62728', '#1f77b4'])

    # Create a flat list of all rectangle artists for direct access
    cell_rects = [ax.add_patch(plt.Rectangle((x - 0.5, y - 0.5), 1, 1)) for y in range(N) for x in range(N)]
    status_text = ax.text(0.5, 1.02, '', transform=ax.transAxes, ha='center', fontsize=12)

    # Set initial grid state
    initial_state_flat = states[0].flatten()
    for i, rect in enumerate(cell_rects):
        rect.set_facecolor(cmap(initial_state_flat[i]))
        rect.set_edgecolor('none') # No borders for speed

    # Track which rectangles have borders to avoid unnecessary iterations
    bordered_rects = set()
    
    def update(frame_idx: int):
        current_state = states[frame_idx]
        next_state = states[frame_idx + 1]

        # Fix: Show the correct iteration number (the result state)
        result_iter_num = start_iteration_offset + frame_idx + 1
        status_text.set_text(f"Result after Iteration {result_iter_num}")

        # --- Core Optimization: Find differences and update only changed cells ---
        diff_indices = np.where(current_state.flatten() != next_state.flatten())[0]

        # Reset borders from previous frame (only for rects that had borders)
        for rect_idx in bordered_rects:
            cell_rects[rect_idx].set_linewidth(0)
            cell_rects[rect_idx].set_edgecolor('none')
        bordered_rects.clear()

        # Update colors and highlight changes
        next_state_flat = next_state.flatten()
        for idx in diff_indices:
            rect = cell_rects[idx]
            new_color = cmap(next_state_flat[idx])
            rect.set_facecolor(new_color)
            rect.set_edgecolor('yellow') # Highlight the change
            rect.set_linewidth(3)
            bordered_rects.add(idx)

        # Add progress updates (fixed to handle floating point precision)
        progress = (frame_idx + 1) / total_frames * 100
        if frame_idx % max(1, total_frames // 5) == 0 and frame_idx > 0:
            print(f"Animation progress: {int(progress)}% complete", file=sys.stderr)

        return cell_rects + [status_text]

    try:
        Writer = animation.writers['ffmpeg']
        # Use the fastest possible preset for ffmpeg
        writer = Writer(fps=fps, metadata=dict(artist='Hyper-Optimized Ising Model'), bitrate=-1,
                        extra_args=['-preset', 'ultrafast', '-crf', '28'])
        anim = animation.FuncAnimation(fig, update, frames=total_frames, blit=True, interval=1000//fps)
        anim.save(output_filename, writer=writer, dpi=dpi)
        print(f"✓ Saved animation: {output_filename}", file=sys.stderr)
        
        # Display the video in the notebook
        if os.path.exists(output_filename):
            display(Video(output_filename, embed=True, width=600, height=600))
        
    except Exception as e:
        print(f"✗ Error saving animation: {e}", file=sys.stderr)
        print("  Ensure you have ffmpeg installed and accessible in your system's PATH.", file=sys.stderr)
    finally:
        plt.close(fig)

# ---------------------------------------------------------------------
# Batch Processing Controller
# ---------------------------------------------------------------------

def batch_animation_processor(
    *base_filenames: str,
    animation_fps: int = 5,
    start_iteration: int = 0,
    end_iteration: int = 19,
    animation_dpi: int = 72, # Lowered DPI for max speed
    force_reprocess: bool = False
):
    """Main controller for batch processing with a specific iteration range."""
    print(f"\n--- Starting Hyper-Optimized Animation Batch for Iterations {start_iteration}-{end_iteration} ---", file=sys.stderr)

    for base_name in base_filenames:
        csv_path = Path(f"{base_name}.csv")
        npz_file_name = f"{base_name}_iter_{start_iteration}_to_{end_iteration}.npz"
        npz_path = csv_path.parent / npz_file_name
        output_file = Path(f"{base_name}_anim_hyper_iter_{start_iteration}_to_{end_iteration}.mp4")

        if not csv_path.exists():
            print(f"✗ Skipping {base_name}: CSV file not found at {csv_path}", file=sys.stderr)
            continue

        if force_reprocess or not npz_path.exists():
            generated_npz = preprocess_csv_to_npz(csv_path, start_iteration=start_iteration, end_iteration=end_iteration)
            if not generated_npz:
                print(f"✗ Skipping {base_name} due to pre-processing failure.", file=sys.stderr)
                continue
        else:
            print(f"-> Found up-to-date .npz file: {npz_path.name}. Skipping pre-processing.", file=sys.stderr)

        create_grid_animation_hyper_optimized(
            npz_path,
            output_filename=str(output_file),
            fps=animation_fps,
            dpi=animation_dpi,
            start_iteration_offset=start_iteration
        )

# ---------------------------------------------------------------------
# Main Execution Block
# ---------------------------------------------------------------------

if __name__ == '__main__':
    simulations_to_process = [
        "BJ=-0.44_Kawasaki_0_tol0.00_scp0.50",
        "BJ=-0.44_DR_1_tol0.50_scp0.50"
    ]

    # --- Minimal Test Call ---
    batch_animation_processor(
        *simulations_to_process,
        animation_fps=10, # Increased FPS for smoother look
        start_iteration=0,
        end_iteration=50, # A slightly longer range to see more changes
        animation_dpi=72,
        force_reprocess=True # Force re-run of pre-processing for this new logic
    )

    print("\n--- Animation Batch Complete ---", file=sys.stderr)


--- Starting Hyper-Optimized Animation Batch for Iterations 0-50 ---
Pre-processing BJ=-0.44_Kawasaki_0_tol0.00_scp0.50.csv (iterations 0-50) -> BJ=-0.44_Kawasaki_0_tol0.00_scp0.50_iter_0_to_50.npz
Successfully pre-processed and saved to BJ=-0.44_Kawasaki_0_tol0.00_scp0.50_iter_0_to_50.npz
Creating hyper-optimized animation for BJ=-0.44_Kawasaki_0_tol0.00_scp0.50_iter_0_to_50.npz
Animation progress: 21% complete
Animation progress: 41% complete
Animation progress: 60% complete
Animation progress: 80% complete
Animation progress: 100% complete
✓ Saved animation: BJ=-0.44_Kawasaki_0_tol0.00_scp0.50_anim_hyper_iter_0_to_50.mp4


Pre-processing BJ=-0.44_DR_1_tol0.50_scp0.50.csv (iterations 0-50) -> BJ=-0.44_DR_1_tol0.50_scp0.50_iter_0_to_50.npz
Successfully pre-processed and saved to BJ=-0.44_DR_1_tol0.50_scp0.50_iter_0_to_50.npz
Creating hyper-optimized animation for BJ=-0.44_DR_1_tol0.50_scp0.50_iter_0_to_50.npz
Animation progress: 21% complete
Animation progress: 41% complete
Animation progress: 60% complete
Animation progress: 80% complete
Animation progress: 100% complete
✓ Saved animation: BJ=-0.44_DR_1_tol0.50_scp0.50_anim_hyper_iter_0_to_50.mp4



--- Animation Batch Complete ---
